# Proxy Design Pattern 

explained using the High-Resolution Image Loading example.

#### The Concept

The Proxy Pattern provides a surrogate or placeholder for another object to control access to it. Why?
- **Lazy Initialization (Virtual Proxy)**: The real object is heavy (e.g., a 500MB image). You don't want to load it until the user actually requests to see it.
- **Access Control (Protection Proxy)**: You want to check permissions before letting the user perform a sensitive action.

Analogy: A Credit Card. It is not the actual money (Cash), but it acts as a proxy for the cash in your bank account.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, both the Real Object and the Proxy must implement the same Interface. The client treats them identically.

#### THE SUBJECT INTERFACE

In [2]:
from abc import ABC, abstractmethod

class Graphic(ABC):
    @abstractmethod
    def display(self) -> None:
        pass

#### THE REAL SUBJECT (Heavy Object)

In [6]:
import time

class RealImage(Graphic):
    def __init__(self, filename: str):
        self.filename = filename
        self._load_from_disk() # Expensive operation happens instantly!

    def _load_from_disk(self):
        print(f"Loading {self.filename} from disk... (This takes 3 seconds)")
        time.sleep(1) # Simulating lag

    def display(self) -> None:
        print(f"Displaying {self.filename}")

#### THE PROXY

In [7]:
class ProxyImage(Graphic):
    """
    The Proxy holds a reference to the RealImage but 
    doesn't create it until 'display()' is actually called.
    """
    def __init__(self, filename: str):
        self.filename = filename
        self._real_image: RealImage = None # Initially Empty

    def display(self) -> None:
        # Lazy Loading Logic
        if self._real_image is None:
            self._real_image = RealImage(self.filename)
        
        # Delegate the actual work
        self._real_image.display()

#### CLIENT CODE

In [9]:
def main():
    print("--- 1. Creating Proxy (No Lag here) ---")
    image = ProxyImage("HighRes_Photo.jpg")
    
    print("\n--- 2. Calling display() (Lag happens now) ---")
    image.display() 

    print("\n--- 3. Calling display() again (Cached - Instant) ---")
    image.display()

if __name__ == "__main__":
    main()

--- 1. Creating Proxy (No Lag here) ---

--- 2. Calling display() (Lag happens now) ---
Loading HighRes_Photo.jpg from disk... (This takes 3 seconds)
Displaying HighRes_Photo.jpg

--- 3. Calling display() again (Cached - Instant) ---
Displaying HighRes_Photo.jpg


## The Pythonic Way

In Python, we don't need explicit interfaces. We can use `__getattr__` to create a generic **"Lazy Wrapper"** that can proxy any class, not just an Image. This is significantly more powerful because you don't need to write a specific **ProxyImage** class for every **RealImage** class.

#### THE REAL CLASS (No change needed)

In [11]:
import time

class RealImage:
    def __init__(self, filename):
        print(f"[System] Loading huge file: {filename}")
        time.sleep(1)
        self.filename = filename

    def show(self):
        print(f"[Display] Rendering {self.filename}")

    def get_info(self):
        return f"File: {self.filename}, Size: 500MB"

#### THE PYTHONIC GENERIC PROXY

In [12]:
class LazyProxy:
    """
    A generic proxy that delays the creation of ANY object 
    until a method or attribute is accessed.
    """
    def __init__(self, cls, *args, **kwargs):
        self._cls = cls
        self._args = args
        self._kwargs = kwargs
        self._instance = None # The placeholder

    def __getattr__(self, name):
        """
        This magic method is ONLY called if the attribute 
        is not found in the Proxy instance itself.
        """
        # 1. Check if the real object exists; if not, create it
        if self._instance is None:
            print("--- Lazy Initialization Triggered ---")
            self._instance = self._cls(*self._args, **self._kwargs)
        
        # 2. Delegate the attribute request to the real object
        return getattr(self._instance, name)

#### CLIENT CODE

In [13]:
def main():
    print("Program Started.")
    
    # We wrap the Class and its arguments inside the Proxy
    # Notice: "Loading huge file" does NOT print yet.
    image = LazyProxy(RealImage, "4k_Wallpaper.png")
    
    print("Proxy created. Doing other work...")
    time.sleep(0.5)
    
    # The moment we touch ANY method (.show), the object creates itself.
    image.show() 
    
    # Subsequent calls go straight to the existing object
    print(image.get_info())

if __name__ == "__main__":
    main()

Program Started.
Proxy created. Doing other work...
--- Lazy Initialization Triggered ---
[System] Loading huge file: 4k_Wallpaper.png
[Display] Rendering 4k_Wallpaper.png
File: 4k_Wallpaper.png, Size: 500MB


#### Key Pythonic Features Used

- `__getattr__`: This is the heart of Python proxying. It effectively says: "If the client asks for a method I don't have (like `show`), initialize the real object and pass the request to it."
- **Generic Class Storage**: The proxy stores `self._cls` (the class type) instead of a hardcoded RealImage instance. This means `LazyProxy` can be used for Database connections, API clients, or File loaders without changing a single line of code.

#### Summary

- **Java-like**: You write a specific Proxy class (`ProxyImage`) for every specific real class. It is explicit and type-safe but verbose.
- **Pythonic**: You write a generic `LazyProxy` once and reuse it everywhere. It uses dynamic dispatch to forward requests.

# Proxy Design Pattern 

applied to a Multi-Region Data Center scenario.

#### The Concept: "Federated Resources"

Imagine you are a user in New York. You connect to your local Data Center (New York). You ask for a specific file.
- **Scenario A (Local)**: The file is in New York. You get it instantly.
- **Scenario B (Remote)**: The file is NOT in New York. The New York server acts as a Proxy. It secretly connects to the London or Tokyo data center, fetches the file, and gives it to you.
- **The Trick**: As the user, you think New York had the file all along. You never knew you talked to London.

## The Classic OOP Way (Java-Style)

We use an Interface to ensure all Data Centers look exactly the same. The Proxy holds references to the other centers.

#### THE INTERFACE

In [15]:
from abc import ABC, abstractmethod
from typing import Optional

class DataCenter(ABC):
    """
    The common interface. Clients only talk to this.
    """
    @abstractmethod
    def get_resource(self, resource_id: str) -> Optional[str]:
        pass

#### THE REAL SUBJECT (Concrete Data Center)

In [16]:
from typing import Dict, Optional

class ConcreteDataCenter(DataCenter):
    def __init__(self, region: str, data: Dict[str, str]):
        self.region = region
        self.storage = data

    def get_resource(self, resource_id: str) -> Optional[str]:
        if resource_id in self.storage:
            print(f"[{self.region}] Found '{resource_id}' locally.")
            return self.storage[resource_id]
        print(f"[{self.region}] '{resource_id}' not found.")
        return None

#### THE PROXY (Federated Data Center)

In [17]:
from typing import List, Optional

class FederatedProxyDataCenter(DataCenter):
    """
    This looks like a normal Data Center, but it has
    connections to other regions.
    """
    def __init__(self, local_dc: ConcreteDataCenter, remotes: List[DataCenter]):
        self.local_dc = local_dc
        self.remotes = remotes

    def get_resource(self, resource_id: str) -> Optional[str]:
        # 1. Try Local First
        print(f"--- Client requests '{resource_id}' from {self.local_dc.region} ---")
        data = self.local_dc.get_resource(resource_id)
        if data:
            return data

        # 2. If missing, Proxy the request to Remotes
        print(f"[{self.local_dc.region}] Proxying request to remote centers...")
        for remote in self.remotes:
            data = remote.get_resource(resource_id)
            if data:
                print(f"[{self.local_dc.region}] Retrieved '{resource_id}' from remote.")
                return data

        print("Error: Resource not found anywhere.")
        return None

#### CLIENT CODE

In [18]:
def main():
    # Setup the physical infrastructure
    dc_ny = ConcreteDataCenter("NY", {"file_A": "Data_A_Content"})
    dc_ldn = ConcreteDataCenter("LDN", {"file_B": "Data_B_Content"})
    dc_tok = ConcreteDataCenter("TOK", {"file_C": "Data_C_Content"})

    # Setup the Proxy (User connects to NY, but NY knows about LDN and TOK)
    # The user only sees "my_server"
    my_server = FederatedProxyDataCenter(local_dc=dc_ny, remotes=[dc_ldn, dc_tok])

    # Case 1: Local Hit
    print(my_server.get_resource("file_A")) 
    print()

    # Case 2: Remote Hit (The Proxy works magic here)
    # User thinks they are asking NY, but NY fetches from London
    print(my_server.get_resource("file_B"))

if __name__ == "__main__":
    main()

--- Client requests 'file_A' from NY ---
[NY] Found 'file_A' locally.
Data_A_Content

--- Client requests 'file_B' from NY ---
[NY] 'file_B' not found.
[NY] Proxying request to remote centers...
[LDN] Found 'file_B' locally.
[NY] Retrieved 'file_B' from remote.
Data_B_Content


## The Pythonic Way

In Python, we can leverage Magic Methods (specifically `__getitem__`) to make our Data Centers behave like `Dictionaries`. Accessing a resource becomes as simple as `datacenter['file_id']`. The Proxy logic is hidden inside the lookup method.

We also use dataclasses for cleaner code.

#### THE DATA CENTER (Acting as Real & Proxy)

In [19]:
from dataclasses import dataclass, field
from typing import Dict, Optional

@dataclass
class SmartDataCenter:
    region: str
    _storage: Dict[str, str] = field(default_factory=dict)
    # The "Peer" connections (Remotes)
    _peers: List["SmartDataCenter"] = field(default_factory=list)

    def connect_peer(self, other_dc: "SmartDataCenter"):
        self._peers.append(other_dc)

    # MAGIC METHOD: __getitem__
    # Allows us to use syntax: server["file_id"]
    def __getitem__(self, resource_id: str) -> str:
        # 1. Check Local
        if resource_id in self._storage:
            return f"[{self.region} LOCAL] {self._storage[resource_id]}"
        
        # 2. Proxy to Peers (Recursive Lookup)
        print(f"   > {self.region} missing '{resource_id}'. Checking peers...")
        for peer in self._peers:
            # We "peek" at the peer safely
            try:
                # We call the peer's __getitem__ directly!
                # Note: In real life, this would be an API call.
                return f"[{self.region} via PROXY -> {peer[resource_id]}]"
            except KeyError:
                continue # Peer didn't have it, try next one

        # 3. Give up
        raise KeyError(f"Resource '{resource_id}' not found in federation.")

#### CLIENT CODE

In [20]:
def main():
    # 1. Create Nodes
    ny  = SmartDataCenter("New York", {"user_data": "User: John Doe"})
    ldn = SmartDataCenter("London",   {"app_logs": "Error: Timeout 500"})
    tok = SmartDataCenter("Tokyo",    {"payment": "Transaction #999"})

    # 2. Wire them up (Mesh Network)
    # NY is connected to London and Tokyo
    ny.connect_peer(ldn)
    ny.connect_peer(tok)

    print("--- Pythonic Data Center Proxy ---")

    try:
        # A. Local Request
        # User asks NY for data that is IN NY.
        print(f"Request 'user_data': {ny['user_data']}")
        print("-" * 40)

        # B. Proxied Request
        # User asks NY for data that is IN LONDON.
        # Syntax looks exactly the same!
        print(f"Request 'app_logs':  {ny['app_logs']}")
        print("-" * 40)

        # C. Deep Proxied Request
        # User asks NY for data in TOKYO
        print(f"Request 'payment':   {ny['payment']}")

    except KeyError as e:
        print(e)

if __name__ == "__main__":
    main()

--- Pythonic Data Center Proxy ---
Request 'user_data': [New York LOCAL] User: John Doe
----------------------------------------
   > New York missing 'app_logs'. Checking peers...
Request 'app_logs':  [New York via PROXY -> [London LOCAL] Error: Timeout 500]
----------------------------------------
   > New York missing 'payment'. Checking peers...
   > London missing 'payment'. Checking peers...
Request 'payment':   [New York via PROXY -> [Tokyo LOCAL] Transaction #999]


#### Key Differences in this Pythonic Version

- **Behaving like a Container**: Instead of calling `server.get_resource("x")`, we treat the server as a data structure: `server["x"]`. This is very common in Python (e.g., accessing config objects, session states).
- **No Explicit "Proxy" Class**: In the OOP version, we had a `ConcreteDataCenter` and a `FederatedProxyDataCenter`. In the Pythonic version, every node is smart. The `SmartDataCenter` class is its own proxy. If it has the data, it's a Server. If it doesn't, it acts as a Proxy and asks its neighbors. This is how real P2P networks (like BitTorrent) or Distributed Hash Tables (DHT) often work.